**Rank, Measure and Freeze the Search Instrument**

This notebook completes the development experiments of the experiment design (`./docs/experiment_design.pdf`, §4): it runs Experiment 3, measures every configuration of Experiments 1–3, and freezes the search instrument before the test split is opened, as the manifest (`./docs/MANIFEST.md`, §5.3, §6 and §7) specifies. Experiment 3 pools the candidates of the three components that `step_5_run_development_experiments.ipynb` carried forward and compares four ranking functions under sum and product aggregation. Every configuration is then measured per fold and query set, with 95 % intervals from a paired bootstrap, and the design's decision rule selects the simplest configuration as good as the best. It reads `./eval/queries_dev.csv`, `./cache/development_runs.json`, the run files of `./cache/runs`, `./cache/exp3_inputs.npz` and the query-encoding times of `./cache/signals/query_latency.csv`, and opens no test file. It outputs:

1. `./results/metrics.csv`: one row per configuration, fold (1–5 and their mean), query set (QS1–QS5 and their macro-average) and subset, with R@10, MRR, unrelated@10, the empty rate, bootstrap intervals and latency
2. `./results/per_query/<config_id>.csv`: every query's rank of the first relevant person, R@10, unrelated@10, latency and displayed top ten
3. `./results/k4_coefficients.csv`: the standardised coefficients of the learned scorer K4, fitted on all five folds
4. `./results/frozen_config.json`: the selected configuration, the runner-up and the reference with every parameter, and the decision table

**1. Configuration and Data**

This cell defines the settings, reads the development queries and the outputs of `step_5_run_development_experiments.ipynb`, and asserts that every person they name is a development person.

1. `queries`: each query's set, fold, target, relevant persons and normalised tokens, in the order of every run file
2. the index-token arrays of `exp3_inputs.npz`: each token's person and field, in the column order of its similarity arrays, from which each person's given-name and surname segments are rebuilt
3. `development`: every configuration of Experiments 1–2 with its tuned parameters, and the components carried into Experiment 3

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and data  —  development queries, runs and Experiment 3 inputs
#
# A run holds, for every query, the ids and scores of its 50 best persons,
# its display cut-off τ_d and the milliseconds its matching took. The
# query-encoding time of each generator comes from step_4_compute_name_signals
# and is added once per generator a configuration uses.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import hashlib
import json
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
EVAL_DIR       = Path("eval")                     # development queries
CACHE_DIR      = Path("cache")                    # runs of step_5_run_development_experiments
RESULTS_DIR    = Path("results")                  # metrics and the frozen config
SEED           = 333               # bootstrap resampling
QUERY_SETS     = ("QS1", "QS2", "QS3", "QS4", "QS5")
FOLDS          = (1, 2, 3, 4, 5)
TOP_K          = 50                # persons kept per query
MAX_UNRELATED  = 3.0               # display cut-off constraint on unrelated@10
TAU_STEPS      = 41                # display cut-offs tried, from 0 to the top score
LAMBDA_EXP2    = 0.9               # dictionary penalty of the Experiment 2 components
LAMBDAS        = (0.6, 0.7, 0.8, 0.9)     # K2's penalty grid
RRF_K          = 60                # reciprocal rank fusion constant
K4_C_VALUES    = (0.1, 1.0, 10.0)  # K4's L2 regularisation grid
BOOTSTRAP      = 1000              # paired bootstrap resamples, clustered by target
LATENCY_BUDGET = 500.0             # p95 milliseconds allowed by the design
METRIC_COLUMNS = ["experiment", "config_id", "config_hash", "generator", "dictionary",
                  "placement", "vocab", "retrieval", "ranker", "aggregation", "params",
                  "split", "fold", "query_set", "subset", "n_queries", "r_at_10", "r_at_10_lo",
                  "r_at_10_hi", "mrr", "mrr_lo", "mrr_hi", "unrelated_at_10", "empty_rate",
                  "latency_p50_ms", "latency_p95_ms", "carried_forward", "status", "notes"]
RULE           = "─" * 72          # frames every printed report


def read_csv(path: Path) -> list[dict]:
    """Read the CSV file `path` into a list of dictionaries."""
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def write_csv(path: Path, rows: list[dict], columns: list[str] | None = None) -> None:
    """Write `rows` to the CSV file `path`, with `columns` (default: the first row's keys)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=columns or list(rows[0]), lineterminator="\n",
                                extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


development = json.loads((CACHE_DIR / "development_runs.json").read_text(encoding="utf-8"))
with np.load(CACHE_DIR / "exp3_inputs.npz") as _stored:
    inputs = {name: _stored[name] for name in _stored.files}   # decompressed once, in memory
person_ids = [str(p) for p in inputs["person_ids"]]
person_index = {pid: i for i, pid in enumerate(person_ids)}
query_rows = read_csv(EVAL_DIR / "queries_dev.csv")
assert [q["query_id"] for q in query_rows] == list(inputs["query_ids"]), "queries misaligned"
assert all(set(json.loads(q["relevant_person_ids"])) <= set(person_index) for q in query_rows), \
    "a person outside the development split"
query_ids = [q["query_id"] for q in query_rows]
query_set_index = np.array([QUERY_SETS.index(q["set"]) for q in query_rows])
query_fold = np.array([int(q["fold"]) for q in query_rows])
query_target = np.array([person_index[json.loads(q["target_person_ids"])[0]] for q in query_rows])
query_length = np.array([np.mean([len(t["norm"]) for t in json.loads(q["query_tokens"])])
                         for q in query_rows])
RELEVANT = np.zeros((len(query_rows), len(person_ids)), dtype=bool)
for _i, _query in enumerate(query_rows):
    RELEVANT[_i, [person_index[p] for p in json.loads(_query["relevant_person_ids"])]] = True

tok_person, tok_surname = inputs["tok_person"], inputs["tok_surname"]
seg_start = np.flatnonzero(np.r_[True, (np.diff(tok_person) != 0) | (np.diff(tok_surname) != 0)])
seg_person, seg_surname = tok_person[seg_start], tok_surname[seg_start]
row_start = inputs["row_start"]
encode_latency = defaultdict(dict)
for _row in read_csv(CACHE_DIR / "signals" / "query_latency.csv"):
    encode_latency[_row["generator"]][_row["query_id"]] = float(_row["ms"])
COMPONENTS = development["exp3"]["components"]
K4_SIGNALS = development["exp3"]["k4_signals"]
print(f"✓ Configuration and data loaded ({len(query_rows)} queries, {len(person_ids)} persons, "
      f"{len(development['configurations'])} configurations of Experiments 1–2, "
      f"components {', '.join(COMPONENTS)}).")

✓ Configuration and data loaded (1743 queries, 507 persons, 27 configurations of Experiments 1–2, components exp2_P3_E4_D3_archive_dict_cutoff, exp2_P3_E1_D3_archive_dict_cutoff, exp1_L3).


**2. Scoring, Tuning and Metrics**

This cell defines the measurement shared by every configuration:

1. **Person scoring**: for each query token, a person's given-name (surname) score is its best similarity among its given-name (surname) tokens; a one-token query scores `max(g, s)`, a longer one the best `agg(g_i, s_j)` over pairs of different tokens, with `agg` the sum (halved, so that full-name and one-token scores share the [0, 1] scale one display cut-off needs) or the product
2. **Tuning**: a fold's parameters and display cut-off `τ_d` are chosen on the other four folds' queries, maximising the macro-average R@10 with at most `MAX_UNRELATED` unrelated persons among the first ten shown
3. **Metrics** (manifest §5.3): R@10, MRR, unrelated@10 and the empty rate; the primary score is the macro-average R@10 over QS1–QS5, averaged over the five folds; latency is the query encoding plus the matching and ranking; intervals come from `BOOTSTRAP` paired resamples of the queries, clustered by target person and identical for every configuration
4. `measure` turns a configuration's per-query answers into its metric rows and per-query file

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Scoring, tuning and metrics  —  shared by every configuration
#
# The bootstrap draws target persons with replacement once, and every
# configuration is measured on the same draws, so that differences between
# configurations are paired.
# ─────────────────────────────────────────────────────────────────────────────
if "RELEVANT" not in globals():
    raise RuntimeError("Run the 'Configuration and Data' cell first.")


def field_scores(similarities: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return per query token every person's given-name and surname scores."""
    maxima = np.maximum.reduceat(similarities, seg_start, axis=1)
    given, surname = (np.zeros((len(similarities), len(person_ids))) for _ in range(2))
    given[:, seg_person[~seg_surname]] = maxima[:, ~seg_surname]
    surname[:, seg_person[seg_surname]] = maxima[:, seg_surname]
    return given, surname


def aggregate(given: np.ndarray, surname: np.ndarray, aggregation: str) -> np.ndarray:
    """Return every person's score: max(g, s) for one token, else the best
    agg(g_i, s_j) over pairs of different query tokens.

    The sum is halved, which leaves every ranking unchanged but keeps full-name
    scores on the [0, 1] scale of one-token scores: one display cut-off serves
    both, where an unscaled sum (up to 2) would hide every one-token query.
    """
    if len(given) == 1:
        return np.maximum(given[0], surname[0])
    combine = {"sum": lambda g, s: (g + s) / 2, "product": np.multiply}
    pairs = combine[aggregation](given[:, None, :], surname[None, :, :])
    pairs[np.arange(len(given)), np.arange(len(given))] = 0
    return pairs.max(axis=(0, 1))


def top_persons(scores: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return the TOP_K best persons with a score above zero (ties by person id)."""
    candidates = np.flatnonzero(scores > 0)
    ranked = candidates[np.lexsort((candidates, -scores[candidates]))][:TOP_K]
    ids, values = np.full(TOP_K, -1), np.zeros(TOP_K)
    ids[:len(ranked)], values[:len(ranked)] = ranked, scores[ranked]
    return ids, values


def metrics_at(run: dict, taus: np.ndarray, rows: np.ndarray) -> dict[str, np.ndarray]:
    """Return per query (of `rows`) and per display cut-off the R@10, reciprocal
    rank, unrelated@10 and emptiness of the run's displayed list."""
    ids, scores = run["ids"][rows], run["scores"][rows]
    relevant = np.take_along_axis(RELEVANT[rows], ids.clip(0), axis=1) & (ids >= 0)
    shown = ((scores[:, :, None] >= taus) & (scores[:, :, None] > 0)).sum(axis=1)
    shown10 = np.minimum(shown, 10)
    cumulative = np.concatenate([np.zeros((len(ids), 1), int), relevant.cumsum(axis=1)], axis=1)
    hits = np.take_along_axis(cumulative, shown10, axis=1)
    first = np.where(relevant.any(axis=1), relevant.argmax(axis=1) + 1, TOP_K + 1)[:, None]
    return {"r10": hits / np.minimum(10, RELEVANT[rows].sum(axis=1))[:, None],
            "rr": np.where(first <= shown, 1.0 / first, 0.0),
            "unrelated": shown10 - hits, "empty": (shown == 0).astype(float)}


def macro(values: np.ndarray, rows: np.ndarray) -> np.ndarray:
    """Return the macro-average over query sets of `values` (rows × cut-offs)."""
    sets = query_set_index[rows]
    return np.mean([values[sets == s].mean(axis=0) for s in np.unique(sets)], axis=0)


def choose(runs: list[dict], taus: np.ndarray, rows: np.ndarray) -> tuple[int, float]:
    """Choose the grid point and display cut-off with the best macro R@10 on
    `rows`, subject to mean unrelated@10 ≤ MAX_UNRELATED (ties: MRR, then the
    earlier point and the lower cut-off)."""
    best_key, best = None, (0, float(taus[-1]))
    for i, run in enumerate(runs):
        metrics = metrics_at(run, taus, rows)
        r10, mrr = macro(metrics["r10"], rows), macro(metrics["rr"], rows)
        unrelated = metrics["unrelated"].mean(axis=0)
        for j, tau in enumerate(taus):
            feasible = unrelated[j] <= MAX_UNRELATED
            key = (feasible, r10[j] if feasible else -unrelated[j], mrr[j], -i, -j)
            if best_key is None or key > best_key:
                best_key, best = key, (i, float(tau))
    return best


_clusters = np.unique(query_target, return_inverse=True)[1]
_draws = np.random.default_rng(SEED).integers(0, _clusters.max() + 1,
                                              size=(BOOTSTRAP, _clusters.max() + 1))
BOOTSTRAP_WEIGHTS = np.stack([np.bincount(d, minlength=_clusters.max() + 1)
                              for d in _draws])[:, _clusters].astype(float)


def bootstrap_interval(values: np.ndarray, valid: np.ndarray, sets: list[int]):
    """Return the 95 % interval of the macro-average over `sets` of a per-query
    metric (`valid` marks the queries it applies to), from the paired bootstrap."""
    per_set = []
    for s in sets:
        rows = valid & (query_set_index == s)
        per_set.append(BOOTSTRAP_WEIGHTS[:, rows] @ values[rows]
                       / BOOTSTRAP_WEIGHTS[:, rows].sum(axis=1).clip(1e-9))
    lower, upper = np.percentile(np.mean(per_set, axis=0), [2.5, 97.5])
    return float(lower), float(upper)


def fold_mean(values: np.ndarray, valid: np.ndarray, sets: list[int]) -> float:
    """Return the macro-average over `sets` of a per-query metric, per fold, averaged."""
    per_fold = []
    for f in FOLDS:
        means = [values[valid & (query_fold == f) & (query_set_index == s)].mean() for s in sets
                 if np.any(valid & (query_fold == f) & (query_set_index == s))]
        per_fold.append(np.mean(means) if means else np.nan)
    return float(np.nanmean(per_fold))


measured, metric_rows = {}, []


def per_query_values(final: dict) -> dict[str, np.ndarray]:
    """Return each query's R@10, reciprocal rank, unrelated@10 and emptiness at
    its own fold's display cut-off."""
    values = {k: np.zeros(len(query_rows)) for k in ("r10", "rr", "unrelated", "empty")}
    for tau in np.unique(final["tau"]):
        rows = final["tau"] == tau
        for name, array in metrics_at(final, np.array([tau]), rows).items():
            values[name][rows] = array[:, 0]
    return values


def subset_values(final: dict) -> dict[str, dict]:
    """Return the per-query metrics of each subset: `all`, and for QS3/QS4 the
    form-level `given` and `surname` hits and reciprocal ranks."""
    values = per_query_values(final)
    subsets = {"all": values | {"valid": np.ones(len(query_rows), bool)}}
    if "subsets" in final:
        for column, name in enumerate(("given", "surname")):
            ranks = final["subsets"][:, column]
            subsets[name] = {"r10": ((ranks >= 1) & (ranks <= 10)).astype(float),
                             "rr": np.where(ranks > 0, 1 / ranks.clip(1), 0.0),
                             "valid": ranks >= 0}
    return subsets


def measure(spec: dict, final: dict, params: dict, notes: str = "") -> dict:
    """Measure one configuration from its per-query answers and record it.

    Args:
        spec: the configuration's settings (config_id, experiment, generator, ...).
        final: per query, `ids`, `scores`, `tau` (its fold's display cut-off),
            `ms` (matching plus encoding) and optionally `subsets` (QS3/QS4 ranks).
        params: per fold (1–5) and pooled (`mean`), the tuned parameters.
        notes: free text for the metric rows.

    Returns:
        The measurement: per-query values, primary score with interval, p95 latency.
    """
    subsets = subset_values(final)
    values, every, all_sets = subsets["all"], subsets["all"]["valid"], list(range(5))
    result = {"spec": spec, "values": values, "final": final, "params": params,
              "primary": fold_mean(values["r10"], every, all_sets),
              "primary_ci": bootstrap_interval(values["r10"], every, all_sets),
              "p95_ms": float(np.percentile(final["ms"], 95))}
    measured[spec["config_id"]] = result
    base = {**{k: spec.get(k, "") for k in METRIC_COLUMNS[:10]}, "status": "ok", "notes": notes,
            "split": "dev", "config_hash": hashlib.sha256(
                json.dumps(spec, sort_keys=True).encode()).hexdigest()[:12]}
    for fold in (*FOLDS, "mean"):
        for subset, metrics in subsets.items():
            for set_name in ((*QUERY_SETS, "macro") if subset == "all" else ("QS3", "QS4")):
                row = base | {"fold": fold, "query_set": set_name, "subset": subset,
                              "params": json.dumps(params[fold])}
                if np.any(row_mask(row, metrics)):
                    metric_rows.append(metric_row(row, metrics, result))
    write_per_query(spec["config_id"], final, values)
    return result


def row_mask(row: dict, metrics: dict) -> np.ndarray:
    """Return the queries a metrics.csv row covers: its fold (or all) and query sets."""
    sets = list(range(5)) if row["query_set"] == "macro" else [QUERY_SETS.index(row["query_set"])]
    in_fold = np.ones(len(query_rows), bool) if row["fold"] == "mean" else query_fold == row["fold"]
    return metrics["valid"] & in_fold & np.isin(query_set_index, sets)


def metric_row(row: dict, metrics: dict, result: dict) -> dict:
    """Fill one metrics.csv row from the per-query `metrics` of its subset."""
    rows = row_mask(row, metrics)
    sets = sorted(set(query_set_index[rows]))
    for name, column in (("r10", "r_at_10"), ("rr", "mrr")):
        if row["fold"] == "mean":
            row[column] = fold_mean(metrics[name], metrics["valid"], sets)
            row[f"{column}_lo"], row[f"{column}_hi"] = bootstrap_interval(
                metrics[name], metrics["valid"], sets)
        else:
            row[column] = float(np.mean([metrics[name][rows & (query_set_index == s)].mean()
                                         for s in sets]))
    row["n_queries"] = int(rows.sum())
    if row["subset"] == "all":
        row["unrelated_at_10"] = float(result["values"]["unrelated"][rows].mean())
        row["empty_rate"] = float(result["values"]["empty"][rows].mean())
        row["latency_p50_ms"], row["latency_p95_ms"] = (
            float(np.percentile(result["final"]["ms"][rows], q)) for q in (50, 95))
    else:
        row["notes"] = "form-level R@10 and MRR of the field's query token"
    return row


def write_per_query(config_id: str, final: dict, values: dict) -> None:
    """Write `./results/per_query/<config_id>.csv`: each query's displayed top ten."""
    rows = []
    for i, query_id in enumerate(query_ids):
        shown = [(p, s) for p, s in zip(final["ids"][i], final["scores"][i])
                 if p >= 0 and s >= final["tau"][i]]
        relevant = [r + 1 for r, (p, _) in enumerate(shown) if RELEVANT[i, p]]
        rows.append({"query_id": query_id, "set": QUERY_SETS[query_set_index[i]],
                     "fold": int(query_fold[i]),
                     "first_relevant_rank": relevant[0] if relevant else "",
                     "r_at_10": round(float(values["r10"][i]), 4),
                     "target_in_top10": int(query_target[i] in [p for p, _ in shown[:10]]),
                     "unrelated_at_10": int(values["unrelated"][i]),
                     "latency_ms": round(float(final["ms"][i]), 2),
                     "top10": ";".join(f"{person_ids[p]}:{s:.3f}" for p, s in shown[:10])})
    write_csv(RESULTS_DIR / "per_query" / f"{config_id}.csv", rows)


def encoding_ms(generators) -> np.ndarray:
    """Return each query's encoding time summed over `generators` (each counted once)."""
    return np.array([sum(encode_latency[g].get(q, 0.0) for g in set(generators))
                     for q in query_ids])


print(f"✓ Scoring, tuning and metrics defined ({BOOTSTRAP} paired resamples over "
      f"{_clusters.max() + 1} target persons).")

✓ Scoring, tuning and metrics defined (1000 paired resamples over 450 target persons).


**3. Experiments 1 and 2: Measurement**

This cell measures every configuration that `step_5_run_development_experiments.ipynb` ran. Each run file already holds every query's answer at its fold's tuned parameters, so this cell adds each query's encoding time to its matching time and computes the metric rows, the QS3/QS4 given-name and surname subsets, and the per-query file. A configuration that could not run gets a `not_run` row with its reason.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiments 1 and 2  —  metrics from the stored runs
# ─────────────────────────────────────────────────────────────────────────────
if "measure" not in globals():
    raise RuntimeError("Run the 'Scoring, Tuning and Metrics' cell first.")

for _config_id, _record in development["configurations"].items():
    _stored = np.load(CACHE_DIR / "runs" / f"{_config_id}.npz")
    assert list(_stored["query_ids"]) == query_ids, f"{_config_id}: run misaligned"
    _generator = _record["spec"]["generator"] or "L1"
    _final = {"ids": _stored["ids"], "scores": _stored["scores"], "tau": _stored["tau"],
              "subsets": _stored["subsets"], "ms": _stored["ms"] + encoding_ms([_generator])}
    _params = {int(f): t | {"tau_d": _record["fold_tau"][f]}
               for f, t in _record["fold_theta"].items()}
    _params["mean"] = _record["pooled_theta"] | {"tau_d": _record["pooled_tau"]}
    measure(_record["spec"], _final, _params, _record["notes"])
for _config_id, (_spec, _status) in development["not_run"].items():
    metric_rows.append({**_spec, "status": _status})

print(f"{RULE}\nExperiments 1 and 2: fold-mean macro R@10 [95 % CI], p95 latency\n{RULE}")
for _config_id in sorted(development["configurations"], key=lambda c: -measured[c]["primary"]):
    _m = measured[_config_id]
    print(f"  {_config_id:<36} {_m['primary']:.3f} [{_m['primary_ci'][0]:.3f}, "
          f"{_m['primary_ci'][1]:.3f}]   {_m['p95_ms']:7.1f} ms")
print(f"{RULE}\n  {len(development['configurations'])} configurations measured, "
      f"{len(development['not_run'])} not run")

────────────────────────────────────────────────────────────────────────
Experiments 1 and 2: fold-mean macro R@10 [95 % CI], p95 latency
────────────────────────────────────────────────────────────────────────
  exp2_P3_E4_D3_archive_dict_cutoff    0.885 [0.864, 0.904]      45.3 ms
  exp2_P3_E1_D3_archive_dict_cutoff    0.876 [0.854, 0.895]      45.2 ms
  exp2_P3_E1_D3_archive_cutoff         0.873 [0.851, 0.893]      44.4 ms
  exp2_P3_E4_D1_archive_dict_cutoff    0.868 [0.846, 0.887]      45.2 ms
  exp2_P3_E4_D2_archive_dict_cutoff    0.865 [0.842, 0.886]      44.8 ms
  exp2_P2_E4_D3                        0.855 [0.835, 0.873]      44.6 ms
  exp2_P3_E1_D2_archive_dict_cutoff    0.851 [0.826, 0.872]      44.8 ms
  exp2_P2_E1_D3                        0.845 [0.823, 0.865]      44.5 ms
  exp2_P3_E1_D1_archive_dict_cutoff    0.845 [0.823, 0.867]      45.1 ms
  exp2_P3_E1_D3_archive_top5           0.844 [0.822, 0.864]      44.5 ms
  exp2_P3_E1_D3_archive_dict_top5      0.829 [0.804, 0.850]

**4. Experiment 3: Ranking and Fusion (RQ3)**

This cell pools the candidates of the three components (Experiment 2's two carried configurations and Experiment 1's best lexical generator, each at its own fold's parameters) and ranks them with each ranking function under sum and product aggregation:

1. **K1**, maximum similarity, and **K2**, route-weighted with the penalty λ of dictionary-derived matches tuned over 0.6–0.9: each index token keeps its best similarity over the three components
2. **K3**, reciprocal rank fusion: each component ranks the persons on its own (as in its experiment), and a person scores Σ 1 / (60 + rank)
3. **K4**, a logistic regression over standardised features of each candidate person: the cosine score of each component embedding, the trigram, Daitch–Mokotoff, Beider–Morse and initial-sound (D4) scores, whether the best match came through the dictionary, whether it was a surname, the query's mean token length, and whether both name parts matched. Its regularisation C is chosen by leave-one-fold-out log loss inside the training folds, whose held-out predictions also tune its display cut-off

It writes the coefficients of K4 fitted on all five folds to `./results/k4_coefficients.csv`.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment 3  —  K1–K4 over the pooled candidates, sum and product
#
# Product aggregation penalises a person who matches on one part of the
# name alone, which sum aggregation lets through at half the score.
# Latency adds the components' matching, each generator's encoding once,
# and the fusion (and, for K4, its signals and model) timed here.
# ─────────────────────────────────────────────────────────────────────────────
if "measured" not in globals():
    raise RuntimeError("Run the 'Experiments 1 and 2' cell first.")

COMPONENT_LAMBDA = [LAMBDA_EXP2 if development["configurations"][c]["spec"]["placement"] else 1.0
                    for c in COMPONENTS]
K4_FEATURES = [*(f"cos_{s}" for s in K4_SIGNALS if s.startswith("E")), "trigram", "dm", "bmpm",
               "initial_sound", "route_dictionary", "best_match_surname", "query_length",
               "both_parts_matched"]
COMPONENT_MS = inputs["ms"][:, :len(COMPONENTS)].sum(axis=1)
COMPONENT_GENERATORS = [development["configurations"][c]["spec"]["generator"] or "L1"
                        for c in COMPONENTS]


def component_arrays(i: int) -> list[tuple[np.ndarray, np.ndarray]]:
    """Return each component's (direct, derived) token similarities for query i."""
    rows = slice(row_start[i], row_start[i + 1])
    return [(inputs[f"direct_{k}"][rows], inputs[f"derived_{k}"][rows])
            for k in range(len(COMPONENTS))]


def fused_scores(ranker: str, aggregation: str, lam: float, i: int) -> np.ndarray:
    """Return every person's score for query i under K1, K2 or K3."""
    arrays = component_arrays(i)
    if ranker in ("K1", "K2"):
        best = np.max([np.maximum(d, lam * r) for d, r in arrays], axis=0)
        return aggregate(*field_scores(best), aggregation)
    total = np.zeros(len(person_ids))
    for (direct, derived), component_lambda in zip(arrays, COMPONENT_LAMBDA):
        scores = aggregate(*field_scores(np.maximum(direct, component_lambda * derived)),
                           aggregation)
        ranked = np.flatnonzero(scores > 0)
        ranked = ranked[np.lexsort((ranked, -scores[ranked]))]
        total[ranked] += 1.0 / (RRF_K + np.arange(1, len(ranked) + 1))
    return total


def fusion_run(ranker: str, aggregation: str, lam: float) -> dict:
    """Answer every query with K1, K2 or K3; time each fusion."""
    run = {"ids": np.full((len(query_rows), TOP_K), -1),
           "scores": np.zeros((len(query_rows), TOP_K)), "ms": np.zeros(len(query_rows))}
    for i in range(len(query_rows)):
        started = time.perf_counter()
        run["ids"][i], run["scores"][i] = top_persons(fused_scores(ranker, aggregation, lam, i))
        run["ms"][i] = (time.perf_counter() - started) * 1000
    return run


def k4_features(aggregation: str, i: int) -> tuple[np.ndarray, np.ndarray]:
    """Return K4's candidate persons for query i and their feature matrix."""
    arrays, rows = component_arrays(i), slice(row_start[i], row_start[i + 1])
    direct, derived = np.max([d for d, _ in arrays], axis=0), np.max([r for _, r in arrays], axis=0)
    given, surname = field_scores(np.maximum(direct, derived))
    candidates = np.flatnonzero(aggregate(given, surname, aggregation) > 0)
    columns = [aggregate(*field_scores(inputs[f"signal_{s}"][rows]), aggregation)
               for s in K4_SIGNALS]
    columns.append(aggregate(*field_scores(derived), aggregation)
                   > aggregate(*field_scores(direct), aggregation))
    columns.append(surname.max(axis=0) >= given.max(axis=0))
    columns.append(np.full(len(person_ids), query_length[i]))
    pairs = (given[:, None, :] > 0) & (surname[None, :, :] > 0)
    pairs[np.arange(len(given)), np.arange(len(given))] = False
    columns.append(pairs.any(axis=(0, 1)) if len(given) > 1 else np.zeros(len(person_ids)))
    return candidates, np.column_stack(columns)[candidates].astype(float)


def k4_fit(features: list, rows: np.ndarray, c_value: float):
    """Fit K4 (standardised features, L2 logistic regression) on the queries `rows`."""
    x = np.concatenate([features[i][1] for i in np.flatnonzero(rows)])
    y = np.concatenate([RELEVANT[i, features[i][0]] for i in np.flatnonzero(rows)])
    scaler = StandardScaler().fit(x)
    return scaler, LogisticRegression(C=c_value, max_iter=2000).fit(scaler.transform(x), y)


def k4_run(features: list, models: dict, ms: np.ndarray) -> dict:
    """Return the run whose query i is scored by `models[fold of query i]`."""
    run = {"ids": np.full((len(query_rows), TOP_K), -1),
           "scores": np.zeros((len(query_rows), TOP_K)), "ms": ms.copy()}
    for i, (candidates, x) in enumerate(features):
        scaler, model = models[query_fold[i]]
        scores, started = np.zeros(len(person_ids)), time.perf_counter()
        if len(candidates):
            scores[candidates] = model.predict_proba(scaler.transform(x))[:, 1]
        run["ms"][i] += (time.perf_counter() - started) * 1000
        run["ids"][i], run["scores"][i] = top_persons(scores)
    return run


def k4_inner(features: list, folds: list[int]) -> tuple[float, dict]:
    """Choose C by leave-one-fold-out log loss over `folds`; return it and the
    held-out models fitted with it, keyed by their held-out fold."""
    best = (np.inf, K4_C_VALUES[0], {})
    for c_value in K4_C_VALUES:
        models = {f: k4_fit(features, np.isin(query_fold, [g for g in folds if g != f]), c_value)
                  for f in folds}
        losses = []
        for f, (scaler, model) in models.items():
            for i in np.flatnonzero((query_fold == f)):
                if len(features[i][0]):
                    p = model.predict_proba(scaler.transform(features[i][1]))[:, 1]
                    y = RELEVANT[i, features[i][0]]
                    p = p.clip(1e-9, 1 - 1e-9)
                    losses.append(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
        if np.mean(losses) < best[0]:
            best = (float(np.mean(losses)), c_value, models)
    return best[1], best[2]


def tune_and_measure(spec: dict, runs: list[dict], thetas: list[dict], k4_tuning=None) -> dict:
    """Tune a configuration per fold and pooled, then measure it.

    Args:
        spec: the configuration's specification.
        runs: one run per grid point, every query answered.
        thetas: the grid points, in the order of `runs`.
        k4_tuning: for K4, per fold the runs to tune on (held-out predictions of the
            training folds) and the C chosen; otherwise the other folds of `runs`.
    """
    train_runs = {f: tuning[0] for f, tuning in k4_tuning.items()} if k4_tuning else None
    taus = np.linspace(0, max(float(r["scores"].max()) for r in runs) or 1, TAU_STEPS)
    choice = {f: choose(train_runs[f] if train_runs else runs, taus, query_fold != f)
              for f in FOLDS}
    pooled = choose(runs, taus, np.ones(len(query_rows), bool))
    final = assemble(runs, choice)
    generators = COMPONENT_GENERATORS + (["L2", "L3", "L4"] if spec["ranker"] == "K4" else [])
    final["ms"] = final["ms"] + COMPONENT_MS + encoding_ms(generators)
    params = {f: thetas[point] | {"tau_d": tau} | ({"c": k4_tuning[f][1]} if k4_tuning else {})
              for f, (point, tau) in choice.items()}
    params["mean"] = thetas[pooled[0]] | {"tau_d": pooled[1]}
    return measure(spec, final, params)


def assemble(runs: list[dict], choice: dict) -> dict:
    """Return the per-query answers of each fold's chosen grid point and cut-off."""
    final = {"ids": np.zeros_like(runs[0]["ids"]), "scores": np.zeros_like(runs[0]["scores"]),
             "ms": np.zeros(len(query_rows)), "tau": np.zeros(len(query_rows))}
    for f, (point, tau) in choice.items():
        rows = query_fold == f
        for key in ("ids", "scores", "ms"):
            final[key][rows] = runs[point][key][rows]
        final["tau"][rows] = tau
    return final


def exp3_spec(ranker: str, aggregation: str) -> dict:
    """Return the specification of one Experiment 3 configuration."""
    return {"config_id": f"exp3_{ranker}_{aggregation}", "experiment": "exp3",
            "generator": "+".join(COMPONENTS), "ranker": ranker, "aggregation": aggregation,
            "components": [development["configurations"][c]["spec"] for c in COMPONENTS]}


k4_rows, k4_models = [], {}
for _aggregation in ("sum", "product"):
    for _ranker in ("K1", "K2", "K3"):
        _lambdas = LAMBDAS if _ranker == "K2" else (1.0,)
        tune_and_measure(exp3_spec(_ranker, _aggregation),
                         [fusion_run(_ranker, _aggregation, lam) for lam in _lambdas],
                         [{"lam": lam} for lam in _lambdas])
    _ms, _features = np.zeros(len(query_rows)), []
    for _i in range(len(query_rows)):
        _started = time.perf_counter()
        _features.append(k4_features(_aggregation, _i))
        _ms[_i] = (time.perf_counter() - _started) * 1000 + inputs["ms"][_i, -1]
    _outer, _tuning = {}, {}
    for _fold in FOLDS:
        _c_value, _inner = k4_inner(_features, [g for g in FOLDS if g != _fold])
        _outer[_fold] = k4_fit(_features, query_fold != _fold, _c_value)
        _tuning[_fold] = ([k4_run(_features, {**_inner, _fold: _outer[_fold]}, _ms)], _c_value)
    _pooled_c, _ = k4_inner(_features, list(FOLDS))
    k4_models[_aggregation] = k4_fit(_features, np.ones(len(query_rows), bool), _pooled_c)
    _coefficients = k4_models[_aggregation][1].coef_[0]
    k4_rows += [{"config_id": f"exp3_K4_{_aggregation}", "feature": name, "coefficient": float(w),
                 "c": _pooled_c} for name, w in zip(K4_FEATURES, _coefficients)]
    tune_and_measure(exp3_spec("K4", _aggregation), [k4_run(_features, _outer, _ms)],
                     [{"c": _pooled_c}], _tuning)
write_csv(RESULTS_DIR / "k4_coefficients.csv", k4_rows)

print(f"{RULE}\nExperiment 3: fold-mean macro R@10 [95 % CI], p95 latency\n{RULE}")
for _config_id in sorted((c for c in measured if c.startswith("exp3")),
                         key=lambda c: -measured[c]["primary"]):
    _m = measured[_config_id]
    print(f"  {_config_id:<16} {_m['primary']:.3f} [{_m['primary_ci'][0]:.3f}, "
          f"{_m['primary_ci'][1]:.3f}]   {_m['p95_ms']:7.1f} ms   {_m['params']['mean']}")
print(f"{RULE}")
for _row in k4_rows:
    print(f"  {_row['config_id']:<16} {_row['feature']:<20} {_row['coefficient']:+.3f}")

────────────────────────────────────────────────────────────────────────
Experiment 3: fold-mean macro R@10 [95 % CI], p95 latency
────────────────────────────────────────────────────────────────────────
  exp3_K2_sum      0.905 [0.886, 0.922]      91.0 ms   {'lam': 0.8, 'tau_d': 0.7500001788139343}
  exp3_K2_product  0.904 [0.885, 0.921]      91.0 ms   {'lam': 0.9, 'tau_d': 0.6250002235174357}
  exp3_K4_sum      0.886 [0.866, 0.905]      96.6 ms   {'c': 10.0, 'tau_d': 0.02461119023172528}
  exp3_K4_product  0.882 [0.861, 0.902]      96.7 ms   {'c': 0.1, 'tau_d': 0.02498676735403307}
  exp3_K3_product  0.878 [0.856, 0.897]      91.2 ms   {'lam': 1.0, 'tau_d': 0.017213114754098358}
  exp3_K1_sum      0.865 [0.845, 0.883]      91.0 ms   {'lam': 1.0, 'tau_d': 0.8500002026557922}
  exp3_K1_product  0.860 [0.841, 0.879]      91.0 ms   {'lam': 1.0, 'tau_d': 0.7250002592802254}
  exp3_K3_sum      0.753 [0.728, 0.777]      91.4 ms   {'lam': 1.0, 'tau_d': 0.04426229508196721}
──────────────────

**5. Freeze**

This cell applies the design's decision rule (§1) to every configuration of Experiments 1–3 and freezes the result before the test split is opened. The selected instrument is the **simplest** configuration whose primary score lies within the 95 % bootstrap interval of the best configuration's and whose 95th-percentile latency is under `LATENCY_BUDGET` ms. Simplicity follows the manifest's order: (1) lexical only, (2) one local embedding, (3) a dictionary configuration, (4) fusion (K1–K3 over the pooled components), (5) the learned scorer K4; within a level, fewer components come first, then the higher score. The runner-up is the best-scoring configuration other than the selected one, and the reference is the current design, L3 with K1. Each keeps the parameters and display cut-off chosen on all five folds together. It writes:

1. `./results/frozen_config.json`: the rule, the best configuration's interval, the decision table and the three frozen configurations, each with its components' parameters, K4's model where used, and a hash
2. `./results/metrics.csv`: every metric row, with `carried_forward` set for the configurations each experiment carried forward

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Freeze  —  the decision rule, step by step, then frozen_config.json
#
# The file holds no measured latency, only whether each configuration is
# under the budget, so a re-run that reaches the same decision writes an
# identical file. step_7_confirm_on_test_split refuses to run unless it is
# committed.
# ─────────────────────────────────────────────────────────────────────────────
if "k4_models" not in globals():
    raise RuntimeError("Run the 'Experiment 3' cell first.")


def simplicity(spec: dict) -> tuple[int, int]:
    """Return (level, components) of a configuration in the manifest's order."""
    if spec["experiment"] == "exp3":
        return (5 if spec["ranker"] == "K4" else 4), len(spec["components"])
    if spec["placement"]:
        return 3, (1 if spec["placement"] == "P1" else 2)
    return (1 if spec["generator"].startswith("L") else 2), 1


def frozen_entry(config_id: str) -> dict:
    """Return the frozen description of one configuration, with every parameter."""
    result = measured[config_id]
    entry = {"spec": result["spec"], "params": result["params"]["mean"]}
    if result["spec"]["experiment"] == "exp3":
        entry["component_params"] = {
            c: development["configurations"][c]["pooled_theta"] for c in COMPONENTS}
        entry["k4_signals"] = K4_SIGNALS
    if result["spec"]["ranker"] == "K4":
        scaler, model = k4_models[result["spec"]["aggregation"]]
        entry["k4_model"] = {"features": K4_FEATURES, "mean": scaler.mean_.tolist(),
                             "scale": scaler.scale_.tolist(), "coef": model.coef_[0].tolist(),
                             "intercept": float(model.intercept_[0])}
    entry["config_hash"] = hashlib.sha256(
        json.dumps(entry, sort_keys=True).encode()).hexdigest()[:12]
    return entry


_best = max(measured, key=lambda c: measured[c]["primary"])
_lower, _upper = measured[_best]["primary_ci"]
decision_table = [{"config_id": c, "level": simplicity(m["spec"])[0],
                   "components": simplicity(m["spec"])[1], "primary": round(m["primary"], 4),
                   "within_ci_of_best": m["primary"] >= _lower,
                   "under_latency_budget": m["p95_ms"] < LATENCY_BUDGET}
                  for c, m in measured.items()]
_eligible = [r for r in decision_table if r["within_ci_of_best"] and r["under_latency_budget"]]
selected = min(_eligible, key=lambda r: (r["level"], r["components"], -r["primary"]))["config_id"]
runner_up = max((c for c in measured if c != selected), key=lambda c: measured[c]["primary"])
frozen = {"rule": "simplest configuration within the 95 % CI of the best primary score with "
                  f"p95 latency < {LATENCY_BUDGET:.0f} ms",
          "best": _best, "best_primary_ci": [round(_lower, 4), round(_upper, 4)],
          "selected": frozen_entry(selected), "runner_up": frozen_entry(runner_up),
          "reference": frozen_entry("exp1_L3"), "decision_table": decision_table}
(RESULTS_DIR / "frozen_config.json").write_text(json.dumps(frozen, indent=1) + "\n",
                                                encoding="utf-8")
_carried = {*development["carried"]["exp1_embeddings"], development["carried"]["exp1_lexical"],
            *development["carried"]["exp2"]}
write_csv(RESULTS_DIR / "metrics.csv",
          [row | {"carried_forward": str(row["config_id"] in _carried).lower()}
           for row in metric_rows], METRIC_COLUMNS)

print(f"{RULE}\nFreeze: best {_best}, primary {measured[_best]['primary']:.3f}, "
      f"95 % CI [{_lower:.3f}, {_upper:.3f}]\n{RULE}")
for _row in sorted(decision_table, key=lambda r: (r["level"], r["components"], -r["primary"])):
    print(f"  {'OK ' if _row in _eligible else '   '}L{_row['level']} "
          f"{_row['config_id']:<36} {_row['primary']:.3f}   within CI "
          f"{str(_row['within_ci_of_best']):<5}  under budget {_row['under_latency_budget']}")
print(f"{RULE}\n  selected {selected}; runner-up {runner_up}; reference exp1_L3")

────────────────────────────────────────────────────────────────────────
Freeze: best exp3_K2_sum, primary 0.905, 95 % CI [0.886, 0.922]
────────────────────────────────────────────────────────────────────────
     L1 exp1_L3                              0.495   within CI False  under budget True
     L1 exp1_L2                              0.446   within CI False  under budget True
     L1 exp1_L4                              0.438   within CI False  under budget True
     L1 exp1_L1                              0.085   within CI False  under budget True
     L2 exp1_E1                              0.742   within CI False  under budget True
     L2 exp1_E4                              0.735   within CI False  under budget True
     L2 exp1_E2                              0.711   within CI False  under budget True
     L2 exp1_E3b                             0.602   within CI False  under budget True
     L2 exp1_E3a                             0.568   within CI False  under budget Tru

**6. Consistency Check**

This cell reads `./results/metrics.csv`, the per-query files and `./results/frozen_config.json` back from disk and checks them independently of the cells above:

1. every configuration of Experiments 1–3 has its rows, and every configuration that ran has its per-query file
2. each configuration's primary score, recomputed from its per-query file with plain arithmetic (R@10 averaged per query set, per fold, then over folds), equals the `mean`/`macro` row of `metrics.csv`
3. the selected configuration is within the best configuration's interval and under the latency budget by the rows of `metrics.csv`, and no simpler configuration is
4. no per-query file names a person outside the development split

It ends with the configurations checked and the result of the check.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  metrics, per-query files and the frozen configuration
# ─────────────────────────────────────────────────────────────────────────────
if "frozen" not in globals():
    raise RuntimeError("Run every cell above first.")


def recomputed_primary(path: Path) -> float:
    """Return the fold-mean macro R@10 recomputed from one per-query file."""
    by_fold = defaultdict(lambda: defaultdict(list))
    for row in read_csv(path):
        by_fold[row["fold"]][row["set"]].append(float(row["r_at_10"]))
    return float(np.mean([np.mean([np.mean(v) for v in sets.values()])
                          for sets in by_fold.values()]))


def freeze_problems(stored: dict, primary: dict, p95: dict) -> list[str]:
    """Check the frozen selection against the decision rule, from the metric rows."""
    eligible = {c for c in primary
                if primary[c] >= stored["best_primary_ci"][0] - 1e-4 and p95[c] < LATENCY_BUDGET}
    selected_id = stored["selected"]["spec"]["config_id"]
    rank = {r["config_id"]: (r["level"], r["components"], -r["primary"])
            for r in stored["decision_table"]}
    problems = [] if selected_id in eligible else [f"{selected_id} breaks the decision rule"]
    return problems + [f"{c} is simpler and eligible" for c in eligible
                       if rank[c] < rank[selected_id]]


_rows = read_csv(RESULTS_DIR / "metrics.csv")
_stored = json.loads((RESULTS_DIR / "frozen_config.json").read_text(encoding="utf-8"))
_macro = {r["config_id"]: r for r in _rows if r["fold"] == "mean" and r["query_set"] == "macro"
          and r["subset"] == "all"}
_expected = {*development["configurations"], *development["not_run"],
             *(f"exp3_{k}_{a}" for k in ("K1", "K2", "K3", "K4") for a in ("sum", "product"))}
_checks = {"configurations": [f"{c}: no rows" for c in _expected - {r["config_id"] for r in _rows}],
           "primary scores": [], "sealed test split": []}
for _config_id, _row in _macro.items():
    _path = RESULTS_DIR / "per_query" / f"{_config_id}.csv"
    if not _path.exists() or abs(recomputed_primary(_path) - float(_row["r_at_10"])) > 1e-4:
        _checks["primary scores"].append(f"{_config_id}: per-query file missing or disagrees")
    _checks["sealed test split"] += [
        f"{_config_id}: names {entry}" for row in (read_csv(_path) if _path.exists() else [])
        for entry in row["top10"].split(";") if entry and entry.split(":")[0] not in person_index]
_checks["decision rule"] = freeze_problems(
    _stored, {c: float(r["r_at_10"]) for c, r in _macro.items()},
    {c: float(r["latency_p95_ms"]) for c, r in _macro.items()})
print(f"{RULE}\nConsistency: {RESULTS_DIR}/ against the development evaluation\n{RULE}")
for _name, _problems in _checks.items():
    print(f"  {'!! ' if _problems else 'OK '}{_name:<20} "
          + (f"{len(_problems)} problem(s)" if _problems else "consistent"))
    for _problem in _problems[:10]:
        print(f"        - {_problem}")
_consistent = not any(_checks.values())
print(f"{RULE}\n  {len(_macro)} configurations measured  "
      f"({'consistent' if _consistent else 'inconsistent'})")
print(f"\n✓ Development experiments complete (selected {_stored['selected']['spec']['config_id']}, "
      f"runner-up {_stored['runner_up']['spec']['config_id']}).")
print(f"  results consistent: {_consistent}")

────────────────────────────────────────────────────────────────────────
Consistency: results/ against the development evaluation
────────────────────────────────────────────────────────────────────────
  OK configurations       consistent
  OK primary scores       consistent
  OK sealed test split    consistent
  OK decision rule        consistent
────────────────────────────────────────────────────────────────────────
  35 configurations measured  (consistent)

✓ Development experiments complete (selected exp3_K2_sum, runner-up exp3_K2_product).
  results consistent: True
